# This's script for generate mockup data

In [8]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ตั้งค่า Seed เพื่อให้ได้ข้อมูลคงที่และสุ่มออกมาเหมือนเดิมทุกครั้งที่รัน
np.random.seed(42)

# กำหนดจำนวนแถวข้อมูลที่ต้องการจำลอง
num_sales = 500
num_products = 15
num_stores = 3
num_pos = 5
num_po_rows = 50

# =========================================================================
# ตารางที่ 3: Product Master Data (ตรงตาม Schema)
# =========================================================================
products = [
    {"product_id": f"P{i:03d}", "product_name": name, "product_taxonomies": category}
    for i, (name, category) in enumerate([
        ("Espresso", "Beverage"), ("Americano", "Beverage"), ("Latte", "Beverage"), 
        ("Green Tea Latte", "Beverage"), ("Thai Milk Tea", "Beverage"), ("Chocolate Frost", "Beverage"),
        ("Croissant", "Bakery"), ("Chocolate Cake", "Bakery"), ("Orange Cake", "Bakery"), 
        ("Strawberry Shortcake", "Bakery"), ("Ham & Cheese Sandwich", "Bakery"),
        ("Tuna Salad", "Food"), ("Spaghetti Carbonara", "Food"), ("Basil Pork Rice", "Food"),
        ("Drinking Water", "Beverage")
    ], start=1)
]
df_product = pd.DataFrame(products)

# =========================================================================
# ตารางที่ 1: Sales Transaction Data (แก้ไข Bug เวลาเรียบร้อย)
# =========================================================================
sales_data = []
# กำหนดเฉพาะวันที่เริ่มต้นหลัก (เวลาเป็น 00:00:00 เพื่อไม่ให้เกิดการบวกเวลาทบ)
start_date = datetime(2026, 5, 20) 

for i in range(1, num_sales + 1):
    # สุ่มจำนวนวันขยับไปข้างหน้า (ข้อมูลในรอบ 1 สัปดาห์)
    random_days = np.random.randint(0, 7)
    dt_day = start_date + timedelta(days=random_days)
    
    # สุ่มเวลาเปิดร้าน 7 โมงเช้า ถึงก่อน 2 ทุ่ม (สุ่มเฉพาะเลขชั่วโมง 7 ถึง 19)
    random_hours = np.random.randint(7, 20) 
    random_minutes = np.random.randint(0, 60)
    random_seconds = np.random.randint(0, 60)
    
    # ใช้ .replace() เพื่อแทนที่เวลาลงไปตรงๆ ป้องกันเวลาหลุดข้ามวันหรือเกินช่วงที่กำหนด
    dt = dt_day.replace(hour=random_hours, minute=random_minutes, second=random_seconds)
    
    prod = np.random.choice(products)
    qty = int(np.random.choice([1, 2, 3], p=[0.7, 0.2, 0.1])) # ซื้อ 1 ชิ้นมีโอกาสสูงสุด
    
    sales_data.append({
        "sales_transaction_id": f"TX{i:05d}",
        "datetime": dt.strftime("%Y-%m-%d %H:%M:%S"),
        "store_id": f"STR{np.random.randint(1, num_stores + 1):02d}",
        "pos_id": f"POS{np.random.randint(1, num_pos + 1):02d}",
        "product_id": prod["product_id"],
        "qty": qty
    })

df_sales = pd.DataFrame(sales_data)

# =========================================================================
# ตารางที่ 2: Purchasing Order Data (ตรงตาม Schema)
# =========================================================================
po_data = []
for i in range(1, num_po_rows + 1):
    prod = np.random.choice(products)
    arr_date = datetime(2026, 5, 18) + timedelta(days=np.random.randint(0, 5))
    
    # สินค้าประเภท Bakery และ Food จะมีอายุสั้นมาก (2-4 วัน) เพื่อให้เกิดปัญหา Food Waste นำไปวิเคราะห์ต่อได้
    if prod["product_taxonomies"] in ["Bakery", "Food"]:
        shelf_life = np.random.choice([2, 3, 4])
    else:
        shelf_life = 60 # เครื่องดื่มมีอายุยาวนานกว่า
        
    exp_date = arr_date + timedelta(days=int(shelf_life))
    
    po_data.append({
        "purchasing_order_id": f"PO{i:04d}",
        "warehouse_id": f"WH{np.random.randint(1, 2):02d}",
        "store_id": f"STR{np.random.randint(1, num_stores + 1):02d}",
        "product_id": prod["product_id"],
        "qty": int(np.random.randint(10, 50)),
        "arrival_date": arr_date.strftime("%Y-%m-%d"),
        "expire_date": exp_date.strftime("%Y-%m-%d")
    })

df_po = pd.DataFrame(po_data)

# =========================================================================
# ตารางที่ 8: Stock Movement Transaction Data (ตรงตาม Schema)
# =========================================================================
stock_data = []
for i in range(1, 100):
    prod = np.random.choice(products)
    move_date = datetime(2026, 5, 18) + timedelta(days=np.random.randint(0, 8))
    stock_data.append({
        "stock_movement_id": f"STK{i:05d}",
        "datetime": move_date.strftime("%Y-%m-%d %H:%M:%S"),
        "store_id": f"STR{np.random.randint(1, num_stores + 1):02d}",
        "product_id": prod["product_id"],
        "qty": int(np.random.choice([5, 10, -2, -5, 20])) # บวกรวมของเข้า ลบคือคัดทิ้ง/ชำรุด
    })
df_stock = pd.DataFrame(stock_data)

# =========================================================================
# บันทึกไฟล์ข้อมูลดิบออกเป็น CSV สำหรับทำงานต่อ
# =========================================================================
df_sales.to_csv("Assets/sales_transaction.csv", index=False)
df_po.to_csv("Assets/purchasing_order.csv", index=False)
df_product.to_csv("Assets/product_master.csv", index=False)
df_stock.to_csv("Assets/stock_movement.csv", index=False)

print("--- บันทึกไฟล์ข้อมูล Mock Data ทั้ง 4 ตารางเรียบร้อยแล้ว! ---")
print(f"ขอบเขตเวลาตาราง Sales อยู่ในช่วง: {df_sales['datetime'].min()} ถึง {df_sales['datetime'].max()}")

--- บันทึกไฟล์ข้อมูล Mock Data ทั้ง 4 ตารางเรียบร้อยแล้ว! ---
ขอบเขตเวลาตาราง Sales อยู่ในช่วง: 2026-05-20 07:11:56 ถึง 2026-05-26 19:42:10
